# 01 — Run the chart-review experiment

**Question:** what does the agent return when we vary the case and the material in the
Task Presentation?

This notebook is the experimental entry point. It runs, or reuses, a small factorial
cohort over synthetic chart cases and two task arms:

- `task_only`: the agent receives the field/output contract but no chart-review policy
  clauses. Material choices can therefore depend on its own judgment.
- `policy_bundle`: the same task plus independently versioned evidence, conflict,
  selection, and proof clauses.

It then reads every run back from Langtrace, reconstructs Decision Episodes with Luna,
selects an analysis only when two reconstruction passes agree on episode alignment,
and projects the result into one Semantica ContextGraph.

## What you should learn

1. How to configure a real experiment without putting keys in the notebook.
2. Which artifacts are created at each boundary.
3. How answers vary by case, arm, and review model.
4. Which observations are experimental results versus hypotheses for later audit.

**Cost/safety:** existing local real-provider runs are reused by default. Set
`ACR_TUTORIAL_MODE=live` explicitly to make paid OpenRouter calls. Run outputs stay
under `runs/`, which Git ignores.


## 1. Locate the repository and choose a mode

`reuse` makes no model or network calls. It uses the checked local cohort that produced
the executed notebook. `live` creates a new cohort. On a fresh clone there is no local
cohort, so the default becomes `live` and the environment checks below fail early if
provider configuration is missing.


In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import asyncio
import json
import os
import subprocess

from IPython.display import Markdown, display
from acr.mvp.langtrace_io import LangtraceClient
from acr.mvp.ledger import SemanticaLedger
from acr.mvp.reconstruct import reconstruct_run
from acr.mvp.reconstruction_llm import AuditedLiteLLM
from acr.mvp.runner import run_patient

START_DIR = Path.cwd().resolve()
ROOT = START_DIR if (START_DIR / "pyproject.toml").is_file() else START_DIR.parent
assert (ROOT / "pyproject.toml").is_file(), "Start Jupyter from the repo or notebooks/"

SEED_ROOT = ROOT / "runs/policy-experiment-20260827"
SEED_LEDGER = SEED_ROOT / "experiment-ledger.json"
LIVE_ROOT = Path(os.environ.get("ACR_TUTORIAL_RUN_ROOT", ROOT / "runs/postdoc-study"))
MODE = os.environ.get(
    "ACR_TUTORIAL_MODE", "reuse" if SEED_LEDGER.is_file() else "live"
).lower()
assert MODE in {"reuse", "live"}
EXPERIMENT_ROOT = SEED_ROOT if MODE == "reuse" else LIVE_ROOT
LEDGER_PATH = SEED_LEDGER if MODE == "reuse" else LIVE_ROOT / "ledger.json"
OUTPUT_ROOT = ROOT / "runs/postdoc-notebook-output"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def one_line(value, limit=96):
    text = " ".join(str("" if value is None else value).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

def display_path(value):
    path = Path(value).resolve()
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def markdown_table(rows, columns):
    def safe(value):
        return one_line(value, 110).replace("|", "/")
    header = "| " + " | ".join(label for _, label in columns) + " |"
    rule = "|" + "|".join("---" for _ in columns) + "|"
    body = [
        "| " + " | ".join(safe(row.get(key, "")) for key, _ in columns) + " |"
        for row in rows
    ]
    return "\n".join([header, rule, *body])

codex_version = subprocess.run(
    ["codex", "--version"], capture_output=True, text=True, check=True
).stdout.strip()
display({"mode": MODE, "codex": codex_version, "ledger": display_path(LEDGER_PATH)})


## 2. Declare the experimental grid

The default `pilot` profile is two deliberately paired cases × two task arms × Luna:

- `SYN0001`: ambiguous cytology **with** a same-day physician impression.
- `SYNX03`: the mirror case, ambiguous cytology **without** that impression.

Set `ACR_TUTORIAL_PROFILE=full` to add Terra as a second acting review model. Luna is
still the reconstruction model so changes in the semantic projection are not
confounded with a different reconstructor.

This is a mechanism demonstration, not a powered accuracy study. Increase cases and
repetitions only after defining the estimand and sampling plan.


In [ ]:
PROFILE = os.environ.get("ACR_TUTORIAL_PROFILE", "pilot").lower()
assert PROFILE in {"pilot", "full"}
CASE_IDS = ["SYN0001", "SYNX03"]
TASK_ARMS = ["task_only", "policy_bundle"]
REVIEW_MODELS = ["openai/gpt-5.6-luna"]
if PROFILE == "full":
    REVIEW_MODELS.append("openai/gpt-5.6-terra")
RECONSTRUCTION_MODEL = "openrouter/openai/gpt-5.6-luna"
RECONSTRUCTION_PASSES = 2
SPEC = ROOT / "assets/specs/STORE.390.date_of_initial_diagnosis.yaml"

plan = [
    {"case_id": case_id, "task_arm": arm, "review_model": model}
    for case_id in CASE_IDS
    for arm in TASK_ARMS
    for model in REVIEW_MODELS
]
display(Markdown(markdown_table(plan, [
    ("case_id", "Case"), ("task_arm", "Task arm"),
    ("review_model", "Acting review model"),
])))
display({"planned_runs": len(plan), "reconstruction_passes_per_run": 2})


## 3. Execute the closed loop, or load the sealed cohort

In `live` mode each row performs the whole boundary:

`Codex App Server → chart tools → local Langtrace → fixed ReAct replay → two Luna
reconstructions → deterministic verification → explicit selection → Semantica`.

No key is passed as a CLI argument or written into an artifact. If the two passes do
not agree on episode alignment, execution stops instead of silently choosing the more
convenient reconstruction.


In [ ]:
if MODE == "live":
    for name in ("OPENROUTER_API_KEY", "LANGTRACE_API_KEY"):
        assert os.environ.get(name), f"{name} is required in live mode"
    langtrace_host = os.environ.get("LANGTRACE_API_HOST", "http://127.0.0.1:3100")
    langtrace_project = os.environ.get("LANGTRACE_PROJECT_ID", "acr_chart_review")
    client = LangtraceClient(
        api_key=os.environ["LANGTRACE_API_KEY"],
        api_host=langtrace_host,
        project_id=langtrace_project,
    )
    ledger = SemanticaLedger(LEDGER_PATH)
    created = []
    for cell in plan:
        run_dir = await asyncio.to_thread(
            run_patient,
            SPEC,
            ROOT / "corpus/patients" / cell["case_id"],
            LIVE_ROOT,
            model=cell["review_model"],
            base_url="https://openrouter.ai/api/v1",
            api_key=os.environ["OPENROUTER_API_KEY"],
            task_arm=cell["task_arm"],
            langtrace_api_key=os.environ["LANGTRACE_API_KEY"],
            langtrace_api_host=langtrace_host,
            langtrace_project_id=langtrace_project,
        )
        runner_meta = json.loads((run_dir / "runner_meta.json").read_text())
        assert runner_meta["langtrace_verified"] is True
        review = client.get_review(runner_meta["langtrace_trace_id"])
        reconstructor = AuditedLiteLLM(
            model=RECONSTRUCTION_MODEL,
            api_key=os.environ["OPENROUTER_API_KEY"],
            temperature=0.0,
        )
        summary = reconstruct_run(
            review,
            ledger,
            reconstructor,
            passes=RECONSTRUCTION_PASSES,
            artifact_dir=run_dir / "analyses",
            reconstructor_identity=RECONSTRUCTION_MODEL,
            max_attempts_per_pass=3,
        )
        assert summary["drift"]["alignment_agrees"] is True, summary["drift"]
        selected = summary["analyses"][0]["analysis_id"]
        ledger.select_analysis(
            review.run_id,
            selected,
            selected_by="postdoc-notebook-01",
            reason="Two Luna passes agreed on atomic episode alignment.",
            provenance="DETERMINISTIC_DERIVED",
        )
        created.append({"run_id": review.run_id, "analysis_id": selected})
    display({"created_runs": len(created), "ledger": display_path(LEDGER_PATH)})
else:
    assert LEDGER_PATH.is_file(), (
        "No reusable cohort is present. Set ACR_TUTORIAL_MODE=live after configuring "
        "OpenRouter and local Langtrace."
    )
    ledger = SemanticaLedger(LEDGER_PATH)
    display({
        "reused_real_provider_ledger": display_path(LEDGER_PATH),
        **ledger.stats(),
    })


## 4. Build one analysis row per run

A run can have several append-only reconstructions. We use the explicit selection when
present; otherwise, for the sealed historical cohort only, we choose the sole/first
reconstruction and mark that fact in the table. Downstream production analysis should
require an explicit selection.

The synthetic corpus includes designer ground truth, so this notebook can display a
correctness check. Real charts will not have that column unless independently
adjudicated.


In [ ]:
analyses_by_run = defaultdict(set)
for node in ledger.graph.find_nodes(node_type="decision"):
    meta = node.get("metadata") or {}
    if meta.get("run_id") and meta.get("analysis_id"):
        analyses_by_run[str(meta["run_id"])].add(str(meta["analysis_id"]))

edge_rows = []
for edge in ledger.graph.edges:
    edge_rows.append(edge.to_dict() if hasattr(edge, "to_dict") else dict(edge))

cohort = []
for run_id, analysis_ids in sorted(analyses_by_run.items()):
    selected = ledger.selected_analysis(run_id)
    analysis_id = selected if selected in analysis_ids else sorted(analysis_ids)[0]
    artifact = ledger.load_analysis_artifact(run_id, analysis_id)
    run_dir = EXPERIMENT_ROOT / run_id
    if not (run_dir / "result.json").is_file():
        continue
    result = json.loads((run_dir / "result.json").read_text())
    case_id = str(artifact.get("patient_id") or run_id.split("_", 2)[1])
    truth_path = ROOT / "corpus/patients" / case_id / "_ground_truth.json"
    truth = json.loads(truth_path.read_text()) if truth_path.is_file() else {}
    gold = (((truth.get("ground_truth") or {}).get(
        "STORE.390.date_of_initial_diagnosis") or {}).get("value"))
    value = result.get("value") or {}
    answer = value.get("date_of_initial_diagnosis") if isinstance(value, dict) else value
    decision_ids = {
        str(node["id"])
        for node in ledger.graph.find_nodes(node_type="decision")
        if (node.get("metadata") or {}).get("run_id") == run_id
        and (node.get("metadata") or {}).get("analysis_id") == analysis_id
    }
    applied_policy_edges = sum(
        edge.get("type") == "APPLIED_POLICY" and edge.get("source_id") in decision_ids
        for edge in edge_rows
    )
    cohort.append({
        "run_id": run_id,
        "case": case_id,
        "arm": artifact.get("task_arm"),
        "review_model": artifact.get("review_model"),
        "analysis_id": analysis_id,
        "selection": "explicit" if selected == analysis_id else "historical fallback",
        "answer": answer,
        "gold": gold,
        "correct": answer == gold if gold is not None else None,
        "episodes": len(artifact.get("episodes") or []),
        "react_cycles": len(artifact.get("cycles") or []),
        "applied_policy_edges": applied_policy_edges,
        "langtrace_events": sum(
            1 for line in (run_dir / "trace.jsonl").read_text().splitlines() if line
        ),
    })

assert cohort, "The ledger contains no run with a local result.json"
display(Markdown(markdown_table(cohort, [
    ("case", "Case"), ("arm", "Arm"), ("review_model", "Review model"),
    ("answer", "Answer"), ("gold", "Synthetic gold"), ("correct", "Match"),
    ("langtrace_events", "Events"), ("react_cycles", "Cycles"),
    ("episodes", "Episodes"), ("applied_policy_edges", "Policy bindings"),
])))


## 5. Inspect variation without overclaiming

The table below groups exact returned values. Variation within the same case and task
arm is a reproducibility signal. A difference between arms is a candidate effect of
the offered policy material, but this small observational cohort cannot isolate that
effect from model identity, run stochasticity, or search-path differences.


In [ ]:
grouped = defaultdict(list)
for row in cohort:
    grouped[(row["case"], row["arm"])].append(row)

variation_rows = []
for (case_id, arm), rows in sorted(grouped.items()):
    distribution = Counter(str(row["answer"]) for row in rows)
    correct = [row["correct"] for row in rows if row["correct"] is not None]
    variation_rows.append({
        "case": case_id,
        "arm": arm,
        "n": len(rows),
        "outcomes": dict(distribution),
        "distinct": len(distribution),
        "gold_matches": f"{sum(correct)}/{len(correct)}" if correct else "not adjudicated",
    })
display(Markdown(markdown_table(variation_rows, [
    ("case", "Case"), ("arm", "Arm"), ("n", "Runs"),
    ("outcomes", "Outcome distribution"), ("distinct", "Distinct outcomes"),
    ("gold_matches", "Gold matches"),
])))

unstable_cells = [row for row in variation_rows if row["distinct"] > 1]
display(Markdown(
    f"**Observed insight:** {len(unstable_cells)} case/arm cell(s) produced more than "
    "one exact answer. Notebook 3 asks whether the divergence can be localized to "
    "a comparable Decision Point rather than only observed at the final answer."
))


## 6. Save a compact handoff for the next notebooks

The full trace, analysis artifact, provenance database, and ContextGraph remain the
authorities. This JSON is only a convenience index; it is not a substitute for them.


In [ ]:
closure = {
    "schema": "acr.postdoc_experiment.v1",
    "mode": MODE,
    "experiment_root": display_path(EXPERIMENT_ROOT),
    "ledger_path": display_path(LEDGER_PATH),
    "cohort": cohort,
    "unstable_case_arm_cells": unstable_cells,
    "interpretation_limits": [
        "small convenience cohort, not a powered comparison",
        "synthetic gold is not a substitute for clinical adjudication",
        "arm differences are not automatically causal policy effects",
    ],
}
closure_path = OUTPUT_ROOT / "01_experiment_summary.json"
closure_path.write_text(json.dumps(closure, ensure_ascii=False, indent=2) + "\n")
assert all((EXPERIMENT_ROOT / row["run_id"] / "trace.jsonl").is_file() for row in cohort)
display(Markdown(
    f"**Notebook 1 closed.** {len(cohort)} real-provider runs are indexed in "
    f"`{display_path(closure_path)}`. Continue to Notebook 2 to inspect how one answer was made."
))
